# Testes Interativos - Camada de Persistencia (SQLAlchemy)

Este notebook testa, de forma interativa, a camada de persistencia do
sistema de gestao de e-commerce/estoque modelado nos diagramas de
classes e E-R do projeto.

Dominio: `Categoria`, `Produto`, `DetalheProduto`, `Pedido`, `ItemPedido`.

Relacionamentos cobertos:
- **1:N** &mdash; `Categoria` &rarr; `Produto`
- **1:1** &mdash; `Produto` &harr; `DetalheProduto`
- **N:M** &mdash; `Produto` &harr; `Pedido` (via `ItemPedido`)

Toda a manipulacao de dados e feita via **SQLAlchemy ORM**; nenhuma
linha de SQL bruto e escrita neste notebook.

O banco utilizado e SQLite (`ecommerce.db`), criado do zero a cada
execucao completa do notebook (ver celula de setup abaixo).


## 1. Instalacao de dependencias

O ambiente virtual que você usar para rodar esse notebook python **DEVE** ter as bibliotecas estabelecidas no requirements.txt

## 2. Setup do ambiente e importacao das classes

A celula abaixo adiciona a pasta `src` ao `sys.path` para permitir a
importacao do pacote `orm_ecommerce` (Database, entidades e
repositorios), sem precisar instalar o pacote formalmente.

Tambem removemos qualquer banco de dados de execucoes anteriores para
garantir que os testes partam de um estado limpo e reprodutivel.

In [1]:
import sys, os
from datetime import date

sys.path.insert(0, os.path.abspath("../src"))

DB_FILE = "ecommerce.db"
if os.path.exists(DB_FILE):
    os.remove(DB_FILE)
    print(f"Banco de dados anterior '{DB_FILE}' removido para reiniciar os testes.")

from orm_ecommerce import (
    Database,
    Categoria, Produto, DetalheProduto, Pedido, ItemPedido,
    CategoriaRepository, ProdutoRepository, DetalheProdutoRepository,
    PedidoRepository, ItemPedidoRepository,
)

print("Classes e repositorios importados com sucesso!")

Classes e repositorios importados com sucesso!


## 3. Criacao do banco de dados e das tabelas

`Database.create_all()` cria as 5 tabelas (`categoria`, `produto`,
`detalhe_produto`, `pedido`, `item_pedido`) a partir dos metadados
declarados nas classes de `models.py` — nenhum `CREATE TABLE` manual
e utilizado.

In [2]:
database = Database(DB_FILE, echo=False)
database.create_all()
print("Banco de dados e tabelas criados com sucesso em:", DB_FILE)

categoria_repo = CategoriaRepository(database)
produto_repo = ProdutoRepository(database)
detalhe_repo = DetalheProdutoRepository(database)
pedido_repo = PedidoRepository(database)
item_repo = ItemPedidoRepository(database)

print("Repositorios instanciados: Categoria, Produto, DetalheProduto, Pedido, ItemPedido")

Banco de dados e tabelas criados com sucesso em: ecommerce.db
Repositorios instanciados: Categoria, Produto, DetalheProduto, Pedido, ItemPedido


## 4. Populando dados sinteticos

Nesta secao criamos um conjunto pequeno e controlado de dados
sinteticos (ficticios), suficiente para exercitar as tres
cardinalidades do modelo. Os dados **nao representam produtos ou
pedidos reais** — nomes, precos e quantidades foram inventados apenas
para fins de teste.

### 4.1. Categorias (dados sinteticos)

Criamos 2 categorias para poder validar o relacionamento **1:N**
(`Categoria` &rarr; `Produto`).

In [3]:
categorias_sinteticas = [
    Categoria(nome_categoria="Perifericos", departamento="Tecnologia"),
    Categoria(nome_categoria="Vestuario", departamento="Moda"),
]

categorias = [categoria_repo.create(c) for c in categorias_sinteticas]

for c in categorias:
    print(f"Categoria criada -> id={c.id_categoria} | {c.nome_categoria} ({c.departamento})")

Categoria criada -> id=1 | Perifericos (Tecnologia)
Categoria criada -> id=2 | Vestuario (Moda)


### 4.2. Produtos (dados sinteticos)

Criamos 4 produtos distribuidos entre as 2 categorias acima. Isso
permite testar, mais adiante, que cada categoria retorna corretamente
a lista de produtos associada (lado "N" da relacao 1:N).

In [4]:
dados_produtos = [
    {"nome": "Mouse Sem Fio",       "preco": 89.90,  "estoque": 50, "categoria": categorias[0]},
    {"nome": "Teclado Mecanico",    "preco": 249.90, "estoque": 30, "categoria": categorias[0]},
    {"nome": "Camiseta Basica",     "preco": 49.90,  "estoque": 100,"categoria": categorias[1]},
    {"nome": "Jaqueta Corta-Vento", "preco": 199.90, "estoque": 20, "categoria": categorias[1]},
]

produtos = []
for d in dados_produtos:
    p = produto_repo.create(
        Produto(
            nome=d["nome"],
            preco_unitario=d["preco"],
            quantidade_estoque=d["estoque"],
            id_categoria=d["categoria"].id_categoria,
        )
    )
    produtos.append(p)
    print(f"Produto criado -> id={p.id_produto} | {p.nome} | categoria={d['categoria'].nome_categoria}")

Produto criado -> id=1 | Mouse Sem Fio | categoria=Perifericos
Produto criado -> id=2 | Teclado Mecanico | categoria=Perifericos
Produto criado -> id=3 | Camiseta Basica | categoria=Vestuario
Produto criado -> id=4 | Jaqueta Corta-Vento | categoria=Vestuario


### 4.3. Detalhes de produto (dados sinteticos, relacao 1:1)

Cada produto recebe **exatamente um** `DetalheProduto`. A restricao
`unique=True` na coluna `id_produto` (ver `models.py`) garante essa
cardinalidade 1:1 no nivel do banco de dados.

In [5]:
detalhes_sinteticos = [
    {"peso": 0.120, "dimensoes": "10x6x4",  "garantia": 12},
    {"peso": 0.950, "dimensoes": "44x14x4", "garantia": 24},
    {"peso": 0.200, "dimensoes": "30x25x2", "garantia": 3},
    {"peso": 0.600, "dimensoes": "40x30x5", "garantia": 6},
]

detalhes = []
for prod, det in zip(produtos, detalhes_sinteticos):
    d = detalhe_repo.create(
        DetalheProduto(
            id_produto=prod.id_produto,
            peso_kg=det["peso"],
            dimensoes_cm=det["dimensoes"],
            garantia_meses=det["garantia"],
        )
    )
    detalhes.append(d)
    print(f"Detalhe criado -> produto_id={d.id_produto} | peso={d.peso_kg}kg | garantia={d.garantia_meses} meses")

Detalhe criado -> produto_id=1 | peso=0.120kg | garantia=12 meses
Detalhe criado -> produto_id=2 | peso=0.950kg | garantia=24 meses
Detalhe criado -> produto_id=3 | peso=0.200kg | garantia=3 meses
Detalhe criado -> produto_id=4 | peso=0.600kg | garantia=6 meses


### 4.4. Pedidos (dados sinteticos)

Criamos 3 pedidos para testar o relacionamento **N:M** com os
produtos.

In [6]:
dados_pedidos = [
    {"data": date(2026, 3, 1), "status": "Processando"},
    {"data": date(2026, 3, 3), "status": "Enviado"},
    {"data": date(2026, 3, 5), "status": "Entregue"},
]

pedidos = [
    pedido_repo.create(Pedido(data_pedido=d["data"], status_pedido=d["status"]))
    for d in dados_pedidos
]

for p in pedidos:
    print(f"Pedido criado -> id={p.id_pedido} | {p.data_pedido} | status={p.status_pedido}")

Pedido criado -> id=1 | 2026-03-01 | status=Processando
Pedido criado -> id=2 | 2026-03-03 | status=Enviado
Pedido criado -> id=3 | 2026-03-05 | status=Entregue


### 4.5. Itens de Pedido (dados sinteticos, relacao N:M)

Cada pedido recebe 2 ou 3 produtos, com quantidades e precos
sinteticos, exercitando o relacionamento **N:M** (um produto presente
em varios pedidos e um pedido contendo varios produtos), implementado
pela classe associativa `ItemPedido`.

In [7]:
itens_sinteticos = [
    # Pedido 1 -> Mouse, Teclado
    (pedidos[0], produtos[0], 2, 89.90),
    (pedidos[0], produtos[1], 1, 249.90),
    # Pedido 2 -> Teclado, Camiseta
    (pedidos[1], produtos[1], 1, 239.90),
    (pedidos[1], produtos[2], 3, 49.90),
    # Pedido 3 -> Camiseta, Jaqueta, Mouse
    (pedidos[2], produtos[2], 2, 44.90),
    (pedidos[2], produtos[3], 1, 199.90),
    (pedidos[2], produtos[0], 1, 79.90),
]

for pedido, produto, quantidade, preco in itens_sinteticos:
    item = item_repo.adicionar_item(
        id_produto=produto.id_produto,
        id_pedido=pedido.id_pedido,
        quantidade=quantidade,
        preco_aplicado=preco,
    )
    print(f"Item criado -> Pedido#{pedido.id_pedido} | {produto.nome} x{quantidade} | preco={item.preco_aplicado}")

print(f"\nTotal de itens de pedido sinteticos criados: {len(itens_sinteticos)}")

Item criado -> Pedido#1 | Mouse Sem Fio x2 | preco=89.90
Item criado -> Pedido#1 | Teclado Mecanico x1 | preco=249.90
Item criado -> Pedido#2 | Teclado Mecanico x1 | preco=239.90
Item criado -> Pedido#2 | Camiseta Basica x3 | preco=49.90
Item criado -> Pedido#3 | Camiseta Basica x2 | preco=44.90
Item criado -> Pedido#3 | Jaqueta Corta-Vento x1 | preco=199.90
Item criado -> Pedido#3 | Mouse Sem Fio x1 | preco=79.90

Total de itens de pedido sinteticos criados: 7


## 5. Testes de leitura e validacao dos relacionamentos

### 5.1. Teste do relacionamento 1:N (`Categoria` &rarr; `Produto`)

Verificamos que cada categoria retorna corretamente os produtos
associados a ela.

In [8]:
for c in categoria_repo.load_all():
    produtos_da_categoria = produto_repo.load_by_categoria(c.id_categoria)
    nomes = ", ".join(p.nome for p in produtos_da_categoria)
    print(f"Categoria {c.nome_categoria}: {len(produtos_da_categoria)} produto(s) -> {nomes}")

assert sum(len(produto_repo.load_by_categoria(c.id_categoria)) for c in categoria_repo.load_all()) == len(produtos)
print("\n[OK] Relacionamento 1:N (Categoria -> Produto) validado com sucesso.")

Categoria Perifericos: 2 produto(s) -> Mouse Sem Fio, Teclado Mecanico
Categoria Vestuario: 2 produto(s) -> Camiseta Basica, Jaqueta Corta-Vento

[OK] Relacionamento 1:N (Categoria -> Produto) validado com sucesso.


### 5.2. Teste do relacionamento 1:1 (`Produto` &harr; `DetalheProduto`)

Verificamos que cada produto possui exatamente um detalhe tecnico
associado e que a busca reversa (detalhe -> produto) tambem
funciona.

In [9]:
for prod in produto_repo.load_all():
    detalhe = detalhe_repo.load_by_produto(prod.id_produto)
    assert detalhe is not None, f"Produto {prod.nome} deveria ter um detalhe!"
    print(f"Produto '{prod.nome}' -> Detalhe: peso={detalhe.peso_kg}kg, dimensoes={detalhe.dimensoes_cm}, garantia={detalhe.garantia_meses} meses")

print("\n[OK] Relacionamento 1:1 (Produto <-> DetalheProduto) validado com sucesso.")

Produto 'Mouse Sem Fio' -> Detalhe: peso=0.120kg, dimensoes=10x6x4, garantia=12 meses
Produto 'Teclado Mecanico' -> Detalhe: peso=0.950kg, dimensoes=44x14x4, garantia=24 meses
Produto 'Camiseta Basica' -> Detalhe: peso=0.200kg, dimensoes=30x25x2, garantia=3 meses
Produto 'Jaqueta Corta-Vento' -> Detalhe: peso=0.600kg, dimensoes=40x30x5, garantia=6 meses

[OK] Relacionamento 1:1 (Produto <-> DetalheProduto) validado com sucesso.


### 5.3. Teste do relacionamento N:M (`Produto` &harr; `Pedido` via `ItemPedido`)

Validamos as duas direcoes da relacao N:M:
- Todos os pedidos em que um produto especifico esta presente.
- Todos os produtos incluidos em um pedido especifico.

In [10]:
print("--- Pedidos por produto ---")
for prod in produto_repo.load_all():
    itens_do_produto = item_repo.load_by_produto(prod.id_produto)
    pedidos_ids = [item.id_pedido for item in itens_do_produto]
    print(f"{prod.nome}: presente nos pedidos {pedidos_ids}")

print("\n--- Produtos por pedido ---")
for pedido in pedido_repo.load_all():
    itens_do_pedido = item_repo.load_by_pedido(pedido.id_pedido)
    produtos_nomes = [
        next(p.nome for p in produtos if p.id_produto == item.id_produto)
        for item in itens_do_pedido
    ]
    print(f"Pedido#{pedido.id_pedido} ({pedido.status_pedido}): {produtos_nomes}")

print("\n[OK] Relacionamento N:M (Produto <-> Pedido via ItemPedido) validado com sucesso.")

--- Pedidos por produto ---
Mouse Sem Fio: presente nos pedidos [1, 3]
Teclado Mecanico: presente nos pedidos [1, 2]
Camiseta Basica: presente nos pedidos [2, 3]
Jaqueta Corta-Vento: presente nos pedidos [3]

--- Produtos por pedido ---
Pedido#1 (Processando): ['Mouse Sem Fio', 'Teclado Mecanico']
Pedido#2 (Enviado): ['Teclado Mecanico', 'Camiseta Basica']
Pedido#3 (Entregue): ['Camiseta Basica', 'Jaqueta Corta-Vento', 'Mouse Sem Fio']

[OK] Relacionamento N:M (Produto <-> Pedido via ItemPedido) validado com sucesso.


## 6. Teste de UPDATE

Atualizamos o preco aplicado de um item de pedido e o status de um
pedido, validando que o metodo `update()` do repositorio persiste as
alteracoes corretamente. Tambem testamos a operacao de baixa de
estoque (`baixar_estoque`) apos uma venda.

In [11]:
# Atualiza o preco do primeiro item sintetico (Mouse no Pedido 1)
primeiro_item = item_repo.load_by_pedido(pedidos[0].id_pedido)[0]
print("Preco antes:", primeiro_item.preco_aplicado)

primeiro_item.preco_aplicado = 79.90
item_repo.update(primeiro_item)

item_atualizado = item_repo.load_from_id(primeiro_item.id_item_pedido)
print("Preco depois:", item_atualizado.preco_aplicado)
assert float(item_atualizado.preco_aplicado) == 79.90

# Atualiza o status do pedido
pedido1 = pedido_repo.load_from_id(pedidos[0].id_pedido)
print("\nStatus antes:", pedido1.status_pedido)
pedido1.status_pedido = "Enviado"
pedido_repo.update(pedido1)
pedido1_atualizado = pedido_repo.load_from_id(pedidos[0].id_pedido)
print("Status depois:", pedido1_atualizado.status_pedido)

# Baixa de estoque apos uma venda
mouse = produtos[0]
estoque_antes = produto_repo.load_from_id(mouse.id_produto).quantidade_estoque
print(f"\nEstoque do Mouse antes da baixa: {estoque_antes}")
produto_repo.baixar_estoque(mouse.id_produto, 2)
estoque_depois = produto_repo.load_from_id(mouse.id_produto).quantidade_estoque
print(f"Estoque do Mouse depois da baixa: {estoque_depois}")
assert estoque_depois == estoque_antes - 2

print("\n[OK] UPDATE (item, pedido e baixa de estoque) validado com sucesso.")

Preco antes: 89.90
Preco depois: 79.90

Status antes: Processando
Status depois: Enviado

Estoque do Mouse antes da baixa: 50
Estoque do Mouse depois da baixa: 48

[OK] UPDATE (item, pedido e baixa de estoque) validado com sucesso.


## 7. Teste de DELETE

Removemos um item de pedido sintetico e confirmamos que ele deixa de
aparecer nas consultas, sem afetar as demais entidades (o produto e o
pedido continuam existindo).

In [12]:
itens_pedido3_antes = item_repo.load_by_pedido(pedidos[2].id_pedido)
print("Itens do Pedido#3 antes do delete:", len(itens_pedido3_antes))

item_repo.delete(itens_pedido3_antes[0])

itens_pedido3_depois = item_repo.load_by_pedido(pedidos[2].id_pedido)
print("Itens do Pedido#3 depois do delete:", len(itens_pedido3_depois))

assert len(itens_pedido3_depois) == len(itens_pedido3_antes) - 1
# O produto e o pedido nao sao apagados, apenas o vinculo (item_pedido)
assert pedido_repo.load_from_id(pedidos[2].id_pedido) is not None
print("\n[OK] DELETE validado com sucesso (somente o vinculo foi removido).")

Itens do Pedido#3 antes do delete: 3
Itens do Pedido#3 depois do delete: 2

[OK] DELETE validado com sucesso (somente o vinculo foi removido).


## 8. Resumo final dos dados persistidos

Consulta geral, exibindo a contagem de registros em cada tabela ao
final dos testes.

In [13]:
resumo = {
    "Categoria": len(categoria_repo.load_all()),
    "Produto": len(produto_repo.load_all()),
    "DetalheProduto": len(detalhe_repo.load_all()),
    "Pedido": len(pedido_repo.load_all()),
    "ItemPedido": len(item_repo.load_all()),
}

print("Resumo final dos registros persistidos:")
for entidade, quantidade in resumo.items():
    print(f" - {entidade}: {quantidade} registro(s)")

print("\nTodos os testes da camada de persistencia foram executados com sucesso!")

Resumo final dos registros persistidos:
 - Categoria: 2 registro(s)
 - Produto: 4 registro(s)
 - DetalheProduto: 4 registro(s)
 - Pedido: 3 registro(s)
 - ItemPedido: 6 registro(s)

Todos os testes da camada de persistencia foram executados com sucesso!


## 9. Encerrando a conexao

Ao final dos testes, fechamos o engine para liberar o arquivo de
banco de dados SQLite.

In [14]:
database.engine.dispose()
print(f"Conexao encerrada. Banco de dados sintetico disponivel em: {DB_FILE}")

Conexao encerrada. Banco de dados sintetico disponivel em: ecommerce.db
